# 02 · Conversations – giving Dummy a memory

A model call is **stateless**. Anything that feels like memory is history that somebody sends again. Foundry offers three ways to handle that:

| 🧠 Way | 🔑 You pass | 🗄️ Who keeps the history |
|---|---|---|
| **Manual** | the full message list every time | your code |
| **Response chain** | `previous_response_id` | Foundry (stored responses) |
| **Conversation** | `conversation=<id>` | Foundry (a durable *conversation* object with items) |

Agents (chapter 04) use conversations. Deep dive: [docs/conversations.md](../docs/conversations.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

oai = openai_client()
MODEL = model("chat")
created = {"conversations": [], "responses": []}   # cleaned up at the end

## 1 · No memory

In [ ]:
a = oai.responses.create(model=MODEL, input="My suit is the Mark 7. Remember that.")
b = oai.responses.create(model=MODEL, input="Which suit did I just mention?")
created["responses"] += [a.id, b.id]
print(b.output_text)      # the model has no idea

## 2 · Chain responses with `previous_response_id`
Foundry stores every response (`store=True` is the default). Pointing at the previous one replays its context.

In [ ]:
a = oai.responses.create(model=MODEL, input="My suit is the Mark 7. Remember that.")
b = oai.responses.create(model=MODEL, input="Which suit did I just mention?", previous_response_id=a.id)
created["responses"] += [a.id, b.id]
print(b.output_text)

### Forking
Because each response is stored, you can branch from an **earlier** one – two futures from the same past.

In [ ]:
fork = oai.responses.create(model=MODEL, input="Actually the suit is the Mark 42. Which suit do I have?", previous_response_id=a.id)
created["responses"].append(fork.id)
print("fork   :", fork.output_text)
print("main   :", b.output_text)

## 3 · Conversations
A **conversation** is a durable object holding an ordered list of **items** (messages, tool calls, tool results). Create it once, then pass its id on every call. Foundry appends the new input *and* the model output as items.

In [ ]:
conv = oai.conversations.create(
    items=[{"type": "message", "role": "user", "content": "I am Tony Stark. Keep answers under 20 words."}],
    metadata={"chapter": "02", "topic": "conversations"},
)
created["conversations"].append(conv.id)
print("conversation:", conv.id)

for question in ["Which suit is the best for underwater missions?", "And which one for space?"]:
    r = oai.responses.create(model=MODEL, conversation=conv.id, input=question)
    created["responses"].append(r.id)
    print(f"\nQ: {question}\nA: {r.output_text}")

### Look inside: the items
This is exactly what the portal shows under *Conversations*.

In [ ]:
items = oai.conversations.items.list(conv.id)
for it in items:
    text = ""
    if it.type == "message":
        text = " ".join(getattr(part, "text", "") for part in it.content)[:90]
    print(f"{it.type:10} {getattr(it, 'role', ''):10} {text}")

### Add items yourself
You can seed context (for example a user profile or a retrieved document) without calling the model.

In [ ]:
oai.conversations.items.create(conv.id, items=[
    {"type": "message", "role": "user", "content": "Note: Pepper has banned the color green in the lab."}
])
r = oai.responses.create(model=MODEL, conversation=conv.id, input="What color should I avoid in the lab, and why?")
created["responses"].append(r.id)
print(r.output_text)

## 4 · Don't store: `store=False`
For zero-retention scenarios, ask Foundry not to persist the response. You then carry the context yourself.

In [ ]:
r = oai.responses.create(model=MODEL, input="Give me a one-word codename for a secret mission.", store=False)
print(r.output_text)
try:
    oai.responses.retrieve(r.id)
    print("stored (unexpected)")
except Exception as exc:
    print("not retrievable, as expected:", type(exc).__name__)

## 5 · Retrieve, list and delete
Data persists until you delete it. Clean up what this notebook created.

In [ ]:
print(oai.responses.retrieve(created["responses"][0]).id)
for rid in created["responses"]:
    try: oai.responses.delete(rid)
    except Exception as exc: print("skip", rid, type(exc).__name__)
for cid in created["conversations"]:
    oai.conversations.delete(cid)
print("cleaned up", len(created["responses"]), "responses and", len(created["conversations"]), "conversations")

## 👀 In the portal
Nothing new under **Agents**: conversations are not agents, and without an agent they have no Foundry page. With tracing on (chapter 03) find their traces in **Application Insights**. From chapter 04, an agent's **Traces** tab lists its conversation ids.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)



## Recap
| Situation | Pick |
|---|---|
| Quick follow-up in a script | `previous_response_id` |
| Chat sessions, agents, anything long-lived | `conversation` |
| Branching / what-if | `previous_response_id` from an earlier response |
| No retention allowed | `store=False` and carry history yourself |

> 💡 Next: [03 · Tracing and monitoring](03_tracing_and_monitoring.ipynb) – see these calls in the portal and in Log Analytics.